# 04 — Regenerate thesis figures (List of Figures)

Creates **Fig 1.1** and **Fig 6.1–6.23** under `figures/`.

**Needs** (from prior training on Drive):
- `models/audio/crema_d/classical/features.csv` (+ joblibs for CMs)
- `models/audio/banglaser/classical/features.csv` (+ joblibs)
- Optional DNN: `models/audio/<ds>/dnn/best.pt` + `features.npz`

Point `--models-root` at `models_final` if that is where finals live.

In [1]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/audio-sentiment-analysis")
DRIVE = Path("/content/drive/MyDrive/audio-sentiment-analysis")
if not ROOT.exists():
    ROOT = DRIVE
assert (ROOT / "src").exists(), ROOT
import os, sys
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("cwd", ROOT)

Mounted at /content/drive
cwd /content/drive/MyDrive/audio-sentiment-analysis


In [3]:
from pathlib import Path
import numpy as np, joblib, torch
from sklearn.model_selection import train_test_split
from src.audio.train_dnn import EmotionLSTM
from src.viz import generate_all as g

ROOT = Path("/content/drive/MyDrive/audio-sentiment-analysis")
OUT = ROOT / "figures"
MODELS = ROOT / "models_final"

def plot_dnn(dnn_dir, dataset, title, out_path):
    dnn_dir = Path(dnn_dir)
    npz = dnn_dir / "features_aug.npz"
    if not npz.exists():
        npz = dnn_dir / "features.npz"
    ckpt, le_path = dnn_dir / "best.pt", dnn_dir / "label_encoder.joblib"
    if not (npz.exists() and ckpt.exists() and le_path.exists()):
        print("skip DNN CM, missing files in", dnn_dir)
        return None
    blob = np.load(npz, allow_pickle=True)
    print(npz.name, "keys", blob.files)
    X, y = blob["x"], blob["y"]  # training saved lowercase x
    le = joblib.load(le_path)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    # same normalization as training — do not reuse an old norm.npz
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True) + 1e-8
    X_test = (X_test - mean) / std

    bi = dataset == "banglaser"
    model = EmotionLSTM(
        n_feat=X.shape[-1], n_classes=len(le.classes_),
        bidirectional=bi, hidden=256 if bi else 128,
        layers=2 if bi else 3, dense=128 if bi else 64,
    )
    state = torch.load(ckpt, map_location="cpu")
    model.load_state_dict(state["model"] if isinstance(state, dict) and "model" in state else state)
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_test), 64):
            logits = model(torch.tensor(X_test[i:i+64], dtype=torch.float32))
            preds.append(logits.argmax(1).numpy())
    return g.fig_confusion(y_test, np.concatenate(preds), list(le.classes_), title, Path(out_path))

# Fig 6.22 only (6.1–6.20 already exist)
plot_dnn(
    MODELS / "audio/crema_d/dnn", "crema_d",
    "Fig 6.22 — DNN confusion (CREMA-D)",
    OUT / "fig_6_22_cm_dnn_crema_d.png",
)

# BanglaSER feature plots + RF/XGB matrices; DNN matrix only if that run finished
g.generate_dataset_figures(
    "banglaser",
    MODELS / "audio/banglaser/classical",
    MODELS / "audio/banglaser/dnn",
    OUT,
    skip_dnn=True,
)
plot_dnn(
    MODELS / "audio/banglaser/dnn", "banglaser",
    "Fig 6.23 — DNN confusion (BanglaSER)",
    OUT / "fig_6_23_cm_dnn_banglaser.png",
)
print(sorted(p.name for p in OUT.glob("fig_*.png")))

features_aug.npz keys ['x', 'y']
  wrote /content/drive/MyDrive/audio-sentiment-analysis/figures/fig_6_22_cm_dnn_crema_d.png

=== BanglaSER (banglaser) ===
  wrote /content/drive/MyDrive/audio-sentiment-analysis/figures/fig_6_2_heatmap_banglaser.png
  wrote /content/drive/MyDrive/audio-sentiment-analysis/figures/fig_6_8_loudness_banglaser.png
  wrote /content/drive/MyDrive/audio-sentiment-analysis/figures/fig_6_9_pitch_banglaser.png
  wrote /content/drive/MyDrive/audio-sentiment-analysis/figures/fig_6_10_mfcc_banglaser.png
  wrote /content/drive/MyDrive/audio-sentiment-analysis/figures/fig_6_11_vq_banglaser.png
  wrote /content/drive/MyDrive/audio-sentiment-analysis/figures/fig_6_12_mfcc_pairplot_banglaser.png
  wrote /content/drive/MyDrive/audio-sentiment-analysis/figures/fig_6_14_top10_banglaser.png
  wrote /content/drive/MyDrive/audio-sentiment-analysis/figures/fig_6_16_kde_banglaser.png
  wrote /content/drive/MyDrive/audio-sentiment-analysis/figures/fig_6_19_cm_rf_banglaser.png
  w